In [ ]:
# LegalEase - FAST version (single cell, no pip install, no API key)
import torch, gradio as gr
from threading import Thread
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

# Fast: 0.5B (~1GB). Better quality but slower: "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 350

dev = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if dev == "cuda" else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(dev).eval()
print("Model ready on", dev)

def _stream(system, user):
    if not user or not user.strip():
        yield "Please enter some text first."
        return
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(dev)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    Thread(target=model.generate, kwargs=dict(
        **inputs, streamer=streamer, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)).start()
    text = ""
    for piece in streamer:      # text appears live, no waiting for the full answer
        text += piece
        yield text

def simplify_text(t):
    yield from _stream("You explain legal language in plain, simple English. Use short sentences and bullet points. Highlight obligations, deadlines and risks.", f"Explain this in simple words:\n\n{t}")

def summarize_document(t):
    yield from _stream("You summarize legal documents. Output: 1) Purpose 2) Key parties 3) Key obligations 4) Important dates/amounts 5) Red flags.", t)

def generate_document(kind, details):
    yield from _stream("You draft clear legal document templates. Use placeholders like [NAME] for missing info. Add a short note that this is a draft, not legal advice.", f"Draft a {kind} with these details:\n{details}")

with gr.Blocks(title="LegalEase") as demo:
    gr.Markdown("# LegalEase\nPlain-English legal help. *Not a substitute for a lawyer.*")
    with gr.Tab("Simplify"):
        i1 = gr.Textbox(lines=8, label="Paste legal text"); o1 = gr.Textbox(lines=10, label="Simple explanation")
        gr.Button("Simplify").click(simplify_text, i1, o1)
    with gr.Tab("Summarize"):
        i2 = gr.Textbox(lines=12, label="Paste document"); o2 = gr.Textbox(lines=10, label="Summary")
        gr.Button("Summarize").click(summarize_document, i2, o2)
    with gr.Tab("Generate"):
        t = gr.Dropdown(["Rental Agreement", "NDA", "Employment Contract", "Freelance Contract", "Legal Notice"], label="Document type", value="NDA")
        d = gr.Textbox(lines=5, label="Details (names, dates, amounts, terms)"); o3 = gr.Textbox(lines=14, label="Draft")
        gr.Button("Generate").click(generate_document, [t, d], o3)

# prevent_thread_lock=True -> cell finishes, app keeps running in background
demo.queue().launch(share=True, prevent_thread_lock=True)
